# Machine Learning Cycle - Image Classification
This notebook documents data acquisition, preprocessing, model training, optimization, testing metrics, prediction, and retraining trigger flow.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
sys.path.append(str(project_root / 'src'))
print('Project root:', project_root)

Project root: /Users/michaelkimani/Desktop/Kaggle Folder


## 1. Data Acquisition
Dataset is image-based (non-tabular) and stored in train/test class folders under archive (14).

In [2]:
from preprocessing import resolve_default_paths

paths = resolve_default_paths(project_root)
print('Train dir:', paths.train_dir)
print('Test dir :', paths.test_dir)

Train dir: /Users/michaelkimani/Desktop/Kaggle Folder/archive (14)/seg_train/seg_train
Test dir : /Users/michaelkimani/Desktop/Kaggle Folder/archive (14)/seg_test/seg_test


## 2. Data Processing
Preprocessing steps:
- resize images to fixed size
- convert RGB pixels into normalized vectors
- derive interpretable features: brightness, blue_ratio, green_ratio, texture_strength

In [3]:
from preprocessing import load_split, sample_story_features

x_train, y_train, classes = load_split(paths.train_dir, max_per_class=200)
x_test, y_test, _ = load_split(paths.test_dir, max_per_class=100)
story_x, story_y = sample_story_features(paths.test_dir, classes, samples_per_class=20)

print('Train shape:', x_train.shape)
print('Test shape :', x_test.shape)
print('Classes    :', classes)
print('Feature rows:', story_x.shape[0])

Train shape: (1200, 3072)
Test shape : (600, 3072)
Classes    : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Feature rows: 120


## 3. Model Creation and Optimization
Optimization choices used:
- `LogisticRegression` with regularization (default L2)
- `solver='saga'` to handle multiclass efficiently
- tuned `max_iter` for better convergence
- standardization before classification

In [4]:
from model import train_model, TrainingConfig

result = train_model(
    project_root,
    TrainingConfig(max_train_per_class=500, max_test_per_class=250, max_iter=60)
)
result

/Users/michaelkimani/Desktop/Kaggle Folder/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


{'model_path': '/Users/michaelkimani/Desktop/Kaggle Folder/models/scene_classifier.pkl',
 'metrics_path': '/Users/michaelkimani/Desktop/Kaggle Folder/results/metrics.json',
 'story_path': '/Users/michaelkimani/Desktop/Kaggle Folder/results/feature_story.json',
 'accuracy': 0.48533333333333334}

## 4. Model Testing and Evaluation Metrics
Explicit metrics reported:
- Accuracy
- Precision
- Recall
- F1-score
Additional metric:
- Confusion Matrix

In [5]:
metrics_path = project_root / 'results' / 'metrics.json'
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))

print('Accuracy:', metrics['accuracy'])
report_df = pd.DataFrame(metrics['classification_report']).transpose()
display(report_df[['precision', 'recall', 'f1-score']].head(8))
print('Confusion Matrix size:', len(metrics['confusion_matrix']), 'x', len(metrics['confusion_matrix'][0]))

Accuracy: 0.48533333333333334


,precision,recall,f1-score
buildings,0.342342,0.304000,0.322034
forest,0.732143,0.656000,0.691983
glacier,0.445946,0.528000,0.483516
mountain,0.510870,0.564000,0.536122
sea,0.345622,0.300000,0.321199
street,0.528302,0.560000,0.543689
accuracy,0.485333,0.485333,0.485333
macro avg,0.484204,0.485333,0.483091


Confusion Matrix size: 6 x 6


### Interpretation of Results
The baseline model captures useful class patterns but still confuses visually similar classes.
Main improvements can come from stronger models (CNN or transfer learning), more iterations, and class-balanced tuning.

## 5. Prediction Function Demo

In [6]:
from prediction import predict_image

sample_image = sorted((paths.test_dir / classes[0]).glob('*.jpg'))[0]
prediction_output = predict_image(project_root, sample_image)
prediction_output

{'predicted_class': 'glacier',
 'confidence': 0.7143771052360535,
 'top_3': [{'class': 'glacier', 'probability': 0.7143771052360535},
  {'class': 'buildings', 'probability': 0.21210435032844543},
  {'class': 'sea', 'probability': 0.060546211898326874}]}

## 6. Retraining Trigger
Retraining flow for grading:
1. Upload images through dashboard/API (`/upload-bulk`) and save into `data/uploads/`.
2. Preprocessing is applied through `src/preprocessing.py`.
3. Retrain via `POST /retrain` or `python src/retrain.py`, reusing the established pipeline and regenerating model + metrics artifacts.